In [1]:
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 4.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.4 MB/s eta 0:00:00:00:0100:01


In [2]:
from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
model.eval()
print("ok")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

In [7]:
text = "Morgen te half twee sal een gecomomeerao elgemesne vergadering van de commissie en van do directeuren dor Maatschappij ter bevordering van het natuurkundig onderzoek der Nederlandsche Koloniën worden gehouden te Amsterdam ln Artia (Ingang : Plantage Mlddellaan). Tot de punten van behandeling behooren: 4o. Medodeelingen omtrent de uitkomsten der Siboga-expeditle, met demonstratie, door professor Max Weber. bo. Ulteenietting van snkels resultaten, die tydens de Siboga-expeditie op hydrographlsch gebied verkregen zyn door den heer Tydeman, kaplteln-luitenant ter zee."

labels = ["person", "geographic location", "organization", "historical_event"]

entities = model.predict_entities(text, labels)

print(entities[0])
for entity in entities:
    print(entity["text"], "=>", entity["label"])

{'start': 106, 'end': 192, 'text': 'Maatschappij ter bevordering van het natuurkundig onderzoek der Nederlandsche Koloniën', 'label': 'organization', 'score': 0.9130646586418152}
Maatschappij ter bevordering van het natuurkundig onderzoek der Nederlandsche Koloniën => organization
Amsterdam => geographic location
Plantage Mlddellaan => geographic location
Siboga-expeditle => historical_event
professor Max Weber => person
Siboga-expeditie => historical_event
hydrographlsch gebied => geographic location
heer Tydeman => person


In [17]:
# read file line by line and predict entities for each line
with open("sample_data/output_text.txt", "r") as f:
    for line in f:
        entities = model.predict_entities(line, labels)
        print(line.strip())
        for entity in entities:
            print(entity["text"], "=>", entity["label"])
        print("\n")



Binnenlandsche Nieuwstijdingen. STATEN-GENERAAL.
Binnenlandsche Nieuwstijdingen => organization


STATEN-GENERAAL => organization




EEISSTE 14 AM ER. Zitting van Vrijdag 14 Januarij. De beraadslaging over de begrooting voor de staatsspoorwegen voor 1870 wordt voortgezet. De heer Ruijdecoper van Maarsseveen is van oordeel; dat de verbetering van de haven van Harlingen bij afzonderlijke wet moest worden voorgedragen. Zij ligt niet in de bedoeling der wet van 1860; de post is ontijdig omdat de aansluiting van den noorderspoorweg aan dc Pruissische westbaan niet verzekerd is, en hij twijfelt of' de verbetering der haven haar voor groote schepen toegankelijk zal maken. De heer van Eijsinga betoogt het algemeen belang van de verbetering der derde haven van het rijk, en wijst met betrekking tot het bezwaar over den vorm op het kanaal van Zuidbeveland en andere werken, waarvan de kosten mede in spoorwegbegrootingen zijn opgenomen, omdat zij met de staatsspoorwegen in verband staan. De heer

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

def is_related_to_event(text, event_description, threshold=0.5):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    text_emb = model.encode(text, convert_to_tensor=True)
    event_emb = model.encode(event_description, convert_to_tensor=True)
    similarity = util.cos_sim(text_emb, event_emb).item()
    return similarity >= threshold, similarity

In [ ]:
text = "The expedition departed from Batavia in 1921 and reached New Guinea."
event = "Centraal Nieuw-Guinea Expeditie (1921-1922)"

related, score = is_related_to_event(text, event)
print(related, score)